In [3]:
import pandas as pd
from pathlib import Path
import sys

import os

# Get the current working directory
_script_dir = Path(os.getcwd())

# The rest of your logic
_project_root = _script_dir.parent.parent
if str(_project_root) not in sys.path:
    sys.path.append(str(_project_root))

from src.config import (
    LOCAL_RAW_GAMES_PATH,
    LOCAL_REGULAR_SEASON_GAMES_PATH,
    LOCAL_TEAMS_HISTORY_CITIES_CONFERENCES_PATH,
    LOCAL_GAMES_FEATURES_PATH,
)

from src.data_processing.ingestion import (
    create_teams_history_table,
    parse_raw_games,
    filter_regular_season_games,
    get_nba_season,
)
from src.data_processing.transformation import add_conference
from src.data_processing.features import create_features_tables, merge_features


In [4]:
raw_games_path: Path = None
teams_history_path: Path = None
output_path: Path = None
current_season_year: int = 2024

In [5]:
# Step 1: Create teams history table
print("\n[Step 1/5] Creating teams history table...")
if teams_history_path is None:
    teams_history_path = (
        # Path(__file__).parent.parent.parent
        _project_root
        / "data"
        / "raw"
        / "historical"
        / "TeamsHistoriesConferenceNBA.csv"
    )

teams_history = create_teams_history_table(
    input_file=str(teams_history_path),
    output_file=str(LOCAL_TEAMS_HISTORY_CITIES_CONFERENCES_PATH),
    current_season_year=current_season_year,
)
print(f"✓ Created teams history table with {len(teams_history)} rows")


[Step 1/5] Creating teams history table...
Output saved to /Users/felipeformenti/dev/fformenti/nba_bets/data/processed/teams_history_expanded.csv
✓ Created teams history table with 1527 rows


In [6]:
# Step 2: Parse and filter raw games
print("\n[Step 2/5] Parsing and filtering raw games...")
if raw_games_path is None:
    raw_games_path = LOCAL_RAW_GAMES_PATH

raw_games = pd.read_csv(raw_games_path, parse_dates=["gameDate"])
parsed_games = parse_raw_games(raw_games)
parsed_games["season"] = parsed_games["gameDate"].apply(get_nba_season)
regular_season_games = filter_regular_season_games(parsed_games)
regular_season_games.to_csv(LOCAL_REGULAR_SEASON_GAMES_PATH, index=False)
print(f"✓ Processed {len(regular_season_games)} regular season games")



[Step 2/5] Parsing and filtering raw games...


/var/folders/_2/8lt451812jdgdwl3yhp1bbhh0000gn/T/ipykernel_68716/3491954169.py:6: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_games = pd.read_csv(raw_games_path, parse_dates=["gameDate"])


✓ Processed 65498 regular season games


In [7]:
# Step 3: Add conference information
print("\n[Step 3/5] Adding conference information...")
games_with_conference = add_conference(regular_season_games, teams_history)
print(f"✓ Added conference information to {len(games_with_conference)} games")

# Step 4: Create feature tables
print("\n[Step 4/5] Creating feature tables...")
create_features_tables(games_with_conference)
print("✓ Created all feature tables")

# Step 5: Merge features
print("\n[Step 5/5] Merging features into final table...")
if output_path is None:
    output_path = LOCAL_GAMES_FEATURES_PATH

final_features = merge_features(games_with_conference)
final_features.to_csv(output_path, index=False)
print(f"✓ Created final features table with {len(final_features)} rows")
print(f"✓ Saved to: {output_path}")

print("\n" + "=" * 60)
print("Pipeline completed successfully!")
print("=" * 60)


[Step 3/5] Adding conference information...
✓ Added conference information to 65498 games

[Step 4/5] Creating feature tables...
✓ Created all feature tables

[Step 5/5] Merging features into final table...
✓ Created final features table with 65500 rows
✓ Saved to: /Users/felipeformenti/dev/fformenti/nba_bets/data/processed/games_features.csv

Pipeline completed successfully!


In [8]:
pd.set_option("display.max_columns", None)

In [10]:
1 - final_features["win_bool"].mean()


np.float64(0.6165954198473282)